# Humain in the loop 

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY") 

google_gemni_key = os.getenv("GOOGLE_API_KEY") 

In [3]:
from langchain.chat_models import init_chat_model
llm  = init_chat_model("gemini-2.5-flash", temperature=0.7, verbose=True)
llm

C:\Users\hp\AppData\Local\Temp\ipykernel_30040\2531707593.py:2: DeprecationWarning: Inferred `model_provider='google_vertexai'` from 'gemini-2.5-flash'. This default will change to 'google_genai' in the next major release.To keep current behavior, pass `model_provider='google_vertexai'` (or use the prefix form, e.g. 'google_vertexai:gemini-2.5-flash'); for AI Studio / Gemini API, use 'google_genai' instead.
  llm  = init_chat_model("gemini-2.5-flash", temperature=0.7, verbose=True)


ChatVertexAI(metadata={'lc_versions': {'langchain-core': '1.6.4', 'langchain': '1.3.17'}}, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'structured_output': True}, project='gen-lang-client-0296871858', model_name='gemini-2.5-flash', full_model_name='projects/gen-lang-client-0296871858/locations/us-central1/publishers/google/models/gemini-2.5-flash', client_options=ClientOptions: {'api_endpoint': 'us-central1-aiplatform.googleapis.com', 'client_cert_source': None, 'client_encrypted_cert_source': Non

In [8]:
from typing import Annotated
from langchain_tavily import TavilySearch 
from langchain_core.tools import tool 
from typing_extensions import TypedDict 

from langgraph.checkpoint.memory import MemorySaver 
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages 
from langgraph.prebuilt import ToolNode, tools_condition 

from langgraph.types import Command, interrupt


In [ ]:
class State(TypedDict): 
    messages: Annotated[list, add_messages]


graph_builder = StateGraph(State)

@tool 
def Humain_assitance(query:str)->str:
    """ Requests assistance from human"""
    human_response = interrupt({"query":query})
    return human_response["data"]


tool = TavilySearch(max_results=2)
tools = [Humain_assitance, tool]
llm_with_tools = llm.bind(tools=tools)

def chatbot(state:State):
    message = llm_with_tools.invoke(state["messages"])
    # Because we will be interrupting during tool execution,
    # we disable parallel tool calling to avoid repeating any
    # tool invocations when we resume.
    return {"messages" : [message]}


graph_builder.add_node("chatbot", chatbot)
tool_node = ToolNode(tools = tools)
graph_builder.add_node("tool_node", tool_node)

graph_builder.add_conditional_edges(
    "chatbot",tools_condition
)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("tool_node", "chatbot")




TypeError: 'TavilySearch' object is not callable